# 03 — GR00T Policy Deep Dive

NVIDIA GR00T N1.5/N1.6 support with two modes:
- **Service**: ZMQ client → Docker container running Isaac-GR00T
- **Local**: Load model directly on GPU (requires `isaac-gr00t` package)

This notebook focuses on data configs and the client — no GPU required.

## Data Configurations

`data_configs.json` defines 25+ embodiment configs using `_extends` inheritance.
Each config maps robot sensors to model modality keys.

In [ ]:
from strands_robots.policies.groot.data_config import (
    Gr00tDataConfig, DATA_CONFIG_MAP, load_data_config, create_custom_data_config
)

print(f"{len(DATA_CONFIG_MAP)} configs loaded\n")
for name in sorted(DATA_CONFIG_MAP):
    cfg = DATA_CONFIG_MAP[name]
    print(f"  {name:30s}  video={cfg.video_keys}  state={cfg.state_keys}")

In [ ]:
# Inspect a specific config
cfg = load_data_config("so100_dualcam")
print(f"name:           {cfg.name}")
print(f"video_keys:     {cfg.video_keys}")
print(f"state_keys:     {cfg.state_keys}")
print(f"action_keys:    {cfg.action_keys}")
print(f"language_keys:  {cfg.language_keys}")
print(f"obs_indices:    {cfg.observation_indices}")
print(f"action_indices: {cfg.action_indices} ({len(cfg.action_indices)} steps)")

In [ ]:
# modality_config() returns the dict Isaac-GR00T loaders expect
mc = cfg.modality_config()
for modality, config in mc.items():
    print(f"  {modality:10s}  keys={config.modality_keys}  delta={config.delta_indices}")

## Custom Data Configs

Create configs for new robots — immediately usable via `load_data_config()`.

In [ ]:
custom = create_custom_data_config(
    name="my_3cam_arm",
    video_keys=["video.left", "video.right", "video.overhead"],
    state_keys=["state.arm_joints", "state.gripper"],
    action_keys=["action.arm_joints", "action.gripper"],
    # language_keys defaults to ["annotation.human.task_description"]
    # action_indices defaults to range(16)
)
print(f"registered: {custom.name}")

# Verify it round-trips
assert load_data_config("my_3cam_arm") is custom
print("load_data_config('my_3cam_arm') ✅")

## Observation & Action Mappings

GR00T uses its own internal key names. The mapping system translates between
robot sensor names and model modality keys.

```python
# Explicit mapping (recommended for non-standard robots)
policy = Gr00tPolicy(
    data_config="so100_dualcam",
    model_path="nvidia/GR00T-N1.6-3B",
    observation_mapping={
        "front_cam":     "video.front",       # robot key → model video key
        "wrist_cam":     "video.wrist",
        "joint_position":"state.single_arm",   # robot key → model state key
        "grip_pos":      "state.gripper",
    },
    action_mapping={
        "action.single_arm": "joint_position",  # model key → robot key
        "action.gripper":    "grip_pos",
    },
)
```

Without explicit mapping, auto-inference runs: exact name match first,
then positional fallback with a logged warning.

## MsgSerializer — ZMQ Wire Format

In [ ]:
from strands_robots.policies.groot.client import MsgSerializer
import numpy as np

# Round-trip: dict with numpy arrays + strings → msgpack bytes → dict
data = {
    "test_array": np.random.rand(3, 3).astype(np.float32),
    "test_string": "hello",
}
encoded = MsgSerializer.to_bytes(data)
decoded = MsgSerializer.from_bytes(encoded)

print(f"encoded size: {len(encoded)} bytes")
print(f"array match: {np.allclose(data['test_array'], decoded['test_array'])}")
print(f"string match: {data['test_string'] == decoded['test_string']}")

In [ ]:
# ModalityConfig also serializes cleanly
from strands_robots.policies.groot.data_config import ModalityConfig

mc = ModalityConfig(delta_indices=[0], modality_keys=["front", "wrist"])
data_with_mc = {"config": mc, "array": np.zeros(3, dtype=np.float32)}
rt = MsgSerializer.from_bytes(MsgSerializer.to_bytes(data_with_mc))
print(f"ModalityConfig round-trip: keys={rt['config'].modality_keys}")

## Gr00tInferenceClient

```python
from strands_robots.policies.groot.client import Gr00tInferenceClient

client = Gr00tInferenceClient(host="localhost", port=5555, timeout_ms=15000)

# Optional auth (plaintext warning if host != localhost)
client = Gr00tInferenceClient(host="10.0.0.5", port=5555, api_token="my-token")

client.ping()                              # True/False
actions = client.get_action(observation)   # calls "get_action" endpoint
client.call_endpoint("custom", data={})    # arbitrary endpoints
client.reconnect()                         # close + re-create socket
```

The token is included in every ZMQ request as `{"api_token": ...}`.
Falls back to `GROOT_API_TOKEN` env var if not passed to constructor.

## GR00T Docker Management Tool

```python
from strands_robots import gr00t_inference
from strands import Agent

agent = Agent(tools=[gr00t_inference])

# Start ZMQ service (default)
agent.tool.gr00t_inference(
    action="start", checkpoint_path="/data/model", port=5555,
    data_config="so100_dualcam",
)

# Start HTTP REST service (auto-switches port to 8000)
agent.tool.gr00t_inference(
    action="start", checkpoint_path="/data/model", port=8000,
    data_config="so100_dualcam", http_server=True,
)

# TensorRT acceleration
agent.tool.gr00t_inference(
    action="start", checkpoint_path="/data/model",
    use_tensorrt=True, vit_dtype="fp8", llm_dtype="nvfp4", dit_dtype="fp8",
)

# Other actions: stop, status, list, restart, find_containers
```